# Lab 6 – CRAG-lite bằng LangGraph

Đồ thị: `prepare` (viết lại, tách câu con theo sản phẩm) → `retrieve` → `grade_documents` (LLM chấm liên quan)
→ không có đoạn liên quan: `transform_query` (≤ 2 lần) rồi `retrieve` lại, hết lượt thì `fallback`
→ có: `generate` → `check_grounding` (LLM kiểm tra từng ý) → không đạt: `regenerate` 1 lần → vẫn không đạt: `fallback`.

So sánh với pipeline cố định của Lab 4 (cùng retrieval, cùng prompt sinh) trên 25 câu golden.

In [1]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 230)

from ragbv.config import load_manifest, REPORTS, DATA
from ragbv import chunking as C, store, retrieval as R, generation as G, evaluation as EV
from ragbv.graph import build_graph
from ragbv.golden import load_golden, is_relevant

docs = load_manifest(); chunks = C.load_chunks("structure"); conn = store.connect()
ret = R.Retriever(chunks, docs, conn, "structure")
app = build_graph(ret, docs)
golden = load_golden()
lab4 = {json.loads(l)["id"]: json.loads(l) for l in (DATA / "runs" / "lab4_basic.jsonl").read_text(encoding="utf-8").splitlines()}

## 1. Đồ thị

In [2]:
mermaid = app.get_graph().draw_mermaid()
(REPORTS / "lab6_graph.md").write_text("# Lab 6 – Đồ thị CRAG-lite\n\n```mermaid\n" + mermaid + "\n```\n", encoding="utf-8")
print(mermaid)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	prepare(prepare)
	retrieve(retrieve)
	grade_documents(grade_documents)
	transform_query(transform_query)
	generate(generate)
	check_grounding(check_grounding)
	regenerate(regenerate)
	finish(finish)
	fallback(fallback)
	__end__([<p>__end__</p>]):::last
	__start__ --> prepare;
	check_grounding -.-> fallback;
	check_grounding -.-> finish;
	check_grounding -.-> regenerate;
	generate --> check_grounding;
	grade_documents -.-> fallback;
	grade_documents -.-> generate;
	grade_documents -.-> transform_query;
	prepare --> retrieve;
	regenerate --> check_grounding;
	retrieve --> grade_documents;
	transform_query -.-> fallback;
	transform_query -.-> retrieve;
	fallback --> __end__;
	finish --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Chạy 25 câu

In [3]:
results = {}
for g in golden:
    t0 = time.perf_counter()
    st = app.invoke({"question": g.question}, config={"recursion_limit": 25})
    results[g.id] = {"state": st, "wall_s": time.perf_counter() - t0}
print("xong", len(results))

C:\Users\Legion\Desktop\Job\Paraline\Lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   3%|▎         | 13/393 [00:00<00:03, 124.16it/s]

Loading weights:   7%|▋         | 26/393 [00:00<00:05, 65.03it/s] 

Loading weights:   9%|▊         | 34/393 [00:00<00:06, 55.03it/s]

Loading weights:  10%|█         | 41/393 [00:00<00:06, 51.83it/s]

... (6 outputs left out)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   1%|▏         | 5/391 [00:02<03:07,  2.06it/s]

Loading weights: 100%|██████████| 391/391 [00:02<00:00, 159.59it/s]

xong 25


In [4]:
def llm_calls(log):
    calls = [e for e in log if "usage" in e]
    return len(calls), sum(e["usage"]["prompt_tokens"] + e["usage"]["completion_tokens"] for e in calls)

rows = []
for g in golden:
    st, fin = results[g.id]["state"], results[g.id]["state"]["final"]
    nodes = [e["node"] for e in st["log"]]
    n_calls, toks = llm_calls(st["log"])
    rel_ctx = fin["contexts"]
    rows.append({"id": g.id, "type": g.type, "in_scope": g.in_scope, "route": fin["route"], "status": fin["status"],
                 "transforms": st.get("transforms", 0), "regenerated": st.get("regenerated", False),
                 "grounded": st.get("grounding", {}).get("grounded"), "sub_queries": len(st["queries"]) if "queries" in st else None,
                 "docs_kept": f"{len(st.get('relevant', []))}/{len(st.get('documents', []))}",
                 "gold_in_ctx": any(is_relevant(c.text, c.doc_id, e) for c in rel_ctx for e in g.evidence) if g.in_scope else None,
                 "llm_calls": n_calls + 1, "llm_tokens": toks, "path": " → ".join(nodes)})
table = pd.DataFrame(rows)
table.drop(columns=["path"])

,id,type,in_scope,route,status,transforms,regenerated,grounded,sub_queries,docs_kept,gold_in_ctx,llm_calls,llm_tokens
0,G01,table_lookup,True,answer,tra_loi,0,False,True,1,2/5,True,4,5193
1,G02,table_lookup,True,answer,tra_loi,0,False,True,1,1/5,True,4,4374
2,G03,table_lookup,True,answer,tra_loi,0,False,True,1,3/5,False,4,4772
3,G04,multi_condition,True,answer,tra_loi,0,False,True,1,4/5,True,4,4085
4,G05,fact,True,answer,tra_loi,0,False,True,1,4/5,True,4,5377
5,G06,fact,True,answer,tra_loi,0,False,True,1,4/5,True,4,5452
6,G07,fact,True,answer,tra_loi,0,False,True,1,3/5,True,4,6009
7,G08,table_lookup,True,answer,tra_loi,0,False,True,1,4/5,True,4,6596
8,G09,multi_condition,True,answer,tra_loi,0,False,True,1,3/5,True,4,5285
9,G10,fact,True,answer,tra_loi,0,False,True,1,2/5,True,4,5058


## 3. So với pipeline cố định (Lab 4)
Cùng judge LLM của Lab 5 (rubric 4 nhãn).

In [5]:
cmp = []
for g in golden:
    a6 = results[g.id]["state"]["final"]["answer"]
    a4 = lab4[g.id]["answer"]
    j4 = EV.judge(g.question, g.answer, a4)["label"] if g.in_scope else ("tu_choi" if lab4[g.id]["status"] == "khong_du_thong_tin" else "tra_loi")
    j6 = EV.judge(g.question, g.answer, a6)["label"] if g.in_scope else ("tu_choi" if results[g.id]["state"]["final"]["status"] == "khong_du_thong_tin" else "tra_loi")
    cmp.append({"id": g.id, "type": g.type, "lab4": j4, "lab6": j6})
cmp = pd.DataFrame(cmp)
ins = cmp[cmp.id.str.startswith("G")]
print("Trong phạm vi (judge):"); print(pd.DataFrame({"lab4": ins.lab4.value_counts(), "lab6": ins.lab6.value_counts()}).fillna(0).astype(int))
print("\nNgoài phạm vi từ chối:", (cmp[cmp.id.str.startswith("O")].lab4 == "tu_choi").sum(), "(lab4) /",
      (cmp[cmp.id.str.startswith("O")].lab6 == "tu_choi").sum(), "(lab6) trên 5")
cmp[cmp.lab4 != cmp.lab6]

Trong phạm vi (judge):
         lab4  lab6
dung_du     9    10
thieu_y    11    10

Ngoài phạm vi từ chối: 5 (lab4) / 5 (lab6) trên 5


,id,type,lab4,lab6
13,G14,fact,thieu_y,dung_du


In [6]:
cost = pd.DataFrame({"lab4 (cố định)": {"LLM calls / câu": 2.0, "token TB / câu": None},
                     "lab6 (đồ thị)": {"LLM calls / câu": table.llm_calls.mean(), "token TB / câu": table.llm_tokens.mean()}})
print(table.route.value_counts()); print(table.path.value_counts().head(8))
cost

route
answer      21
fallback     4
Name: count, dtype: int64
path
prepare → retrieve → grade_documents → generate → check_grounding → finish                                                                       20
prepare → retrieve → grade_documents → transform_query → retrieve → grade_documents → transform_query → retrieve → grade_documents → fallback     3
prepare → retrieve → grade_documents → transform_query → retrieve → grade_documents → transform_query → fallback                                  1
prepare → retrieve → grade_documents → transform_query → retrieve → grade_documents → generate → check_grounding → finish                         1
Name: count, dtype: int64


,lab4 (cố định),lab6 (đồ thị)
LLM calls / câu,2.0,4.32
token TB / câu,NaN,5443.52


## 4. Năm câu tiêu biểu: log từng bước

In [7]:
for gid in ["G19", "G03", "G16", "G17", "O02"]:
    st = results[gid]["state"]
    print(f"########## {gid}: {next(g.question for g in golden if g.id == gid)}")
    for e in st["log"]:
        info = {k: v for k, v in e.items() if k not in ("node", "usage")}
        print(f"  [{e['node']}] {json.dumps(info, ensure_ascii=False)[:400]}")
    print("  => " + st["final"]["answer"][:600] + "\n")

########## G19: thoi han bao hiem cua tu ky tai tuc hang thang dai ichi la bao lau, NDBH toi da bao nhieu tuoi
  [prepare] {"rewritten": "Thời hạn bảo hiểm của Tự Ký Tái Tục Hàng Tháng Dai Ichi là bao lâu, Người được bảo hiểm tối đa bao nhiêu tuổi?", "sub_queries": 1}
  [retrieve] {"n": 5}
  [grade_documents] {"kept": 3, "of": 5}
  [generate] {"status": "tra_loi"}
  [check_grounding] {"grounded": true, "unsupported": []}
  [finish] {}
  => Tuổi tối đa của Người được bảo hiểm khi kết thúc Thời hạn bảo hiểm là 76 tuổi [C1].

########## G03: Với quyền lợi thai sản của Manulife Chăm Sóc Sức Khỏe Nâng Cao, chi phí sinh được chi trả như thế nào?
  [prepare] {"rewritten": "Với quyền lợi thai sản của Manulife Chăm Sóc Sức Khỏe Nâng Cao, chi phí sinh được chi trả như thế nào?", "sub_queries": 1}
  [retrieve] {"n": 5}
  [grade_documents] {"kept": 3, "of": 5}
  [generate] {"status": "tra_loi"}
  [check_grounding] {"grounded": true, "unsupported": []}
  [finish] {}
  => Quyền lợi thai sản chỉ được

## 5. Lưu

In [8]:
L = ["# Lab 6 – Câu trả lời của đồ thị CRAG-lite", "", "Sinh tự động. Mỗi câu: đường đi qua đồ thị, câu trả lời, nhãn judge (so với Lab 4).", ""]
for g in golden:
    st = results[g.id]["state"]; r = table.set_index("id").loc[g.id]; c = cmp.set_index("id").loc[g.id]
    L += [f"## {g.id} · {g.type} · route `{r.route}` · judge lab4 `{c.lab4}` → lab6 `{c.lab6}`", "",
          f"**Câu hỏi:** {g.question}", "", f"**Đáp án chuẩn:** {g.answer}", "",
          f"**Đồ thị:** {st['final']['answer']}", "", f"**Lab 4:** {lab4[g.id]['answer']}", "",
          f"*Đường đi:* {r.path}", ""]
    gr = st.get("grounding", {})
    if gr.get("claims"):
        L += ["| Ý | Kết luận grounding |", "|---|---|"] + [f"| {x['claim'][:200]} | {x['verdict']} |" for x in gr["claims"]] + [""]
(REPORTS / "lab6_answers.md").write_text("\n".join(L) + "\n", encoding="utf-8")
out = {"table": table.to_dict("records"), "compare": cmp.to_dict("records"),
       "judge_counts": {"lab4": ins.lab4.value_counts().to_dict(), "lab6": ins.lab6.value_counts().to_dict()},
       "llm_calls_mean": float(table.llm_calls.mean()), "llm_tokens_mean": float(table.llm_tokens.mean()),
       "wall_s": {gid: round(v["wall_s"], 2) for gid, v in results.items()}}
(REPORTS / "lab6_results.json").write_text(json.dumps(out, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print("saved")

saved
